In [ ]:
# Pose landmark detection (Live feed)

In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks.python import vision
from mediapipe.tasks.python import BaseOptions
import time

result_list = []

def res_callback(result, output_image, timestamp_ms):
    result_list.append(result)

from mediapipe.tasks.python import BaseOptions
base_options = BaseOptions(
    model_asset_path="pose_landmarker_lite.task")

model_asset_path="/Users/koshu/Downloads/pose_landmarker_lite.task"

options = vision.PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="/Users/koshu/Downloads/pose_landmarker_lite.task"),
    running_mode= vision.RunningMode.LIVE_STREAM,
    result_callback=res_callback)
landmarker = vision.PoseLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)

while True:
    ret,frame = cap.read()
    if ret == False:
        break
    h, w, _ = frame.shape
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame_rgb = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

    landmarker.detect_async(frame_rgb, time.time_ns() // 1_000_000)

    if result_list:
        for lm in result_list[0].pose_landmarks:
            for each_lm in lm:
                if each_lm.visibility > 0.9:
                    x_each_lm = int(each_lm.x * w)
                    y_each_lm = int(each_lm.y * h)
                    cv2.circle(frame, (x_each_lm, y_each_lm), 8, (0, 255, 255), -1)

        result_list.clear()
    cv2.imshow('Video', frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyALLWindows()

I0000 00:00:1775764298.198093  859067 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1775764298.257432  859072 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775764298.270149  859069 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775764299.905459  859074 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
